# Langbridge SDK Local Runtime Notebook

This notebook uses `LangbridgeClient.local(config_path=...)` against the local runtime defined in `langbridge_config.yml`.

In [ ]:
from pathlib import Path
import sys

EXAMPLE_DIR = Path.cwd()
REPO_ROOT = EXAMPLE_DIR.parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import dotenv
dotenv.load_dotenv(".env")

In [ ]:
import pandas as pd
from langbridge import LangbridgeClient
from setup import setup_database

CONFIG_PATH = EXAMPLE_DIR / "langbridge_config.yml"
setup_database()
client = LangbridgeClient.local(config_path=str(CONFIG_PATH))

## List datasets

In [ ]:
datasets = await client.datasets.list()
pd.DataFrame([item.model_dump(mode="json") for item in datasets.items])

## Run a semantic query


In [ ]:
country_sales = await client.semantic.query(
    "commerce_performance",
    measures=["shopify_orders.net_sales", "shopify_orders.gross_margin"],
    dimensions=["shopify_orders.country"],
    order={"shopify_orders.net_sales": "desc"},
    limit=5,
)
print(pd.DataFrame(country_sales.rows))


## Run dataset SQL

In [ ]:
sql_result = client.sql.query(
    query_scope="dataset",
    query="""
SELECT
    shopify_orders.country,
    SUM(shopify_orders.net_revenue) AS total_net_sales,
    SUM(shopify_orders.gross_margin) AS total_gross_margin
FROM
    shopify_orders
GROUP BY
    shopify_orders.country
ORDER BY
    total_net_sales DESC
LIMIT 5;
    """,
)
pd.DataFrame(sql_result.rows)

## Ask the configured local agent

In [ ]:
answer = await client.agents.ask("Show me top countries by net sales this quarter")
print(answer.text)
print(pd.DataFrame(answer.result["rows"]))